# Unrestricted Hartree Fock Challenges
This notebook will exemplify some issues in the development of the UHF and what/how fixes were implemented.

In [27]:
from pyscf import gto, scf
import numpy as np
from pathlib import Path
from Dev.CSUHF import CS_UHF_ContextClass, CS_UHF
from py_mods.src.SCF.RHF import plot_map

## Incorrect Li SCF convergence
This example shows how a flawed guess can lead to errors. In this particular case, with a $H_core$ guess, the DIIS gets stuck in an incorrect minimum, and converges to a wrong state. However, if we let the regular SCF continue direvtly it bounces to the correct minimum.

In [28]:
# pyscf data
mol_He = gto.M(
    atom='  Li 0 0 0; ', 
    spin=1,
    charge=0, 
    basis='aug-cc-pvqz'
)

kin = mol_He.intor('int1e_kin')
vnuc = mol_He.intor('int1e_nuc')
overlap = mol_He.intor('int1e_ovlp')
eri = mol_He.intor('int2e')

rhf_He = scf.UHF(mol_He)

e_He = rhf_He.kernel()
e_elec = rhf_He.energy_elec()

print(e_elec)

converged SCF energy = -7.43271871704646  <S^2> = 0.75001472  2S+1 = 2.0000147
(np.float64(-7.4327187170464555), np.float64(2.281003741563637))


In [30]:
# Preparation of the context and calculation
Li_context = CS_UHF_ContextClass(overlap, kin, vnuc, eri, n_electrons=3)
Li_context.max_iter = 300
Li_context.conv_type = 'DIIS'
Li_context.p_guess = 'core'
Li_context.verbose = False
Li_context.threshold = 1E-12

In [32]:
Li_UHF_results = CS_UHF(Li_context)
print(f'Converged: {Li_UHF_results.converged} in {Li_UHF_results.iterations}')
print(f'Final energy: {Li_UHF_results.E_UHF}')
print(f'Error in Final energy: {Li_UHF_results.E_UHF-e_elec[0]}')

Converged: True in 109
Final energy: (-7.365043883077648-2.558558142926814e-19j)
Error in Final energy: (0.06767483396880714-2.558558142926814e-19j)


Which is not correct, as it has converged to an incorrect minimum. To fix this problem, we have introduced a RHF guess, that generates a RHF of this system or the cation (to have even electrons) and then starts the UHF calculation from this density:

In [33]:
Li_context.p_guess = 'RHF'
Li_correct_UHF_results = CS_UHF(Li_context)
print(f'Converged: {Li_correct_UHF_results.converged} in {Li_correct_UHF_results.iterations}')
print(f'Final energy: {Li_correct_UHF_results.E_UHF}')
print(f'Error in Final energy: {Li_correct_UHF_results.E_UHF-e_elec[0]}')

Converged: True in 19
Final energy: (-7.432718717046821+1.4961851472783798e-25j)
Error in Final energy: (-3.659295089164516e-13+1.4961851472783798e-25j)
